In [1]:
import numpy as np

In [2]:
def Gauss_Newton(r, x0, itermax, tol, J=None):
    """
    Algoritmo de Gauss-Newton para optimización por mínimos cuadrados no lineales
    o resolución de sistemas no lineales r(x) = 0.
    
    Parámetros:
    -----------
    r : función
        Función vectorial de residuos r(x).
    x0 : array-like
        Vector o punto de estimación inicial.
    itermax : int
        Número máximo de iteraciones.
    tol : float
        Tolerancia para el criterio de parada (norma del residuo o del paso).
    J : función, opcional
        Función que calcula la matriz Jacobiana J(x). Si es None, se aproxima
        numéricamente mediante diferencias finitas.
        
    Retorna:
    --------
    xk : ndarray
        Aproximación de la solución del problema.
    orbita : list
        Lista con el historial de puntos xk visitados.
    error : float
        Norma del residuo final ||r(xk)||.
    """
    # Garantizar vector unidimensional de flotantes
    x0 = np.array(x0, dtype=float).flatten()
    xk = x0.copy()
    orbita = [xk.copy()]
    
    # Función interna para aproximación numérica de la Jacobiana (diferencias finitas)
    def jacobiano_numerico(x, eps=1e-8):
        rx = r(x)
        m = len(rx)
        n = len(x)
        J_mat = np.zeros((m, n))
        for j in range(n):
            x_step = x.copy()
            x_step[j] += eps
            J_mat[:, j] = (r(x_step) - rx) / eps
        return J_mat

    iter_count = 0
    error = np.linalg.norm(r(xk))
    
    while error > tol and iter_count < itermax:
        rx = r(xk)
        
        # Evaluar la matriz Jacobiana analítica o numérica
        Jx = J(xk) if J is not None else jacobiano_numerico(xk)
        
        # Resolver el sistema J(x) * s = -r(x) en sentido de mínimos cuadrados
        # np.linalg.lstsq ofrece mayor estabilidad numérica que la inversión explícita
        sk, _, _, _ = np.linalg.lstsq(Jx, -rx, rcond=None)
        
        # Actualización de la solución
        xk = xk + sk
        orbita.append(xk.copy())
        
        # Criterio de parada basado en la magnitud del paso
        if np.linalg.norm(sk) < tol * (np.linalg.norm(xk) + 1.0):
            error = np.linalg.norm(r(xk))
            break
            
        error = np.linalg.norm(r(xk))
        iter_count += 1
        
    return xk, orbita, error

In [4]:
f = lambda X: np.array([
    -400 * X[0] * (X[1] - X[0]**2) - 2 * (1 - X[0]),
    200 * (X[1] - X[0]**2)
])

x0 = np.array([0.2, 0.2])
xk, orbita, error = Gauss_Newton(f, x0, int(1e3), 1e-6)

print('Solución:', xk)
print('Error:', error)
print('Iteraciones:', len(orbita) - 1)

Solución: [1. 1.]
Error: 4.863152313964054e-08
Iteraciones: 6
